# Hands-on 1 — Halos as Graphs: Node Classification (Cluster vs Void)

**Granada AI School — GNNs for Cosmology**

In this first 1.5h session you will:

1. Load a dark-matter **halo catalog** (positions + masses) from an N-body simulation box.
2. Turn the halo point cloud into a **graph** (nodes = halos, edges = spatial proximity, with periodic boundaries).
3. Define a physically meaningful **node label**: does a halo live in a *cluster* (high-density) or a *void* (low-density) environment?
4. Build and train a **Graph Neural Network (GCN)** to classify each halo.
5. Evaluate, visualise, and experiment.

> **Key idea:** we give the network only per-halo features (mass) plus the *graph structure*. The network must infer the *environment* from the topology of connections. This is the whole point of GNNs — the relationships carry the information.


## Step 0 — Setup

In [ ]:
# Run once (Colab). On a local install you can skip if already present.
!pip -q install torch torch-geometric networkx scipy scikit-learn matplotlib pandas

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.metrics import classification_report, confusion_matrix

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 1 — Get a halo catalog from CAMELS-SAM (via Google Drive)

We use one simulation box from the public **CAMELS-SAM** suite (Rockstar halo catalogs):

`.../Rockstar/CAMELS-SAM/LH/LH_35/Rockstar/out_99.list`  → snapshot 99 = **z = 0** (present day).

The box is **100 Mpc/h** (from the catalog header). The workflow below is meant for **Google Colab**:

1. **Mount your Google Drive.**
2. **Download the catalog once into your Drive** (`MyDrive/GNN_school/data/`). Storing it in your
   Drive means it survives runtime restarts and is not re-downloaded every time.
3. **Load** positions, masses and velocities from the file.

> The z = 0 file is large (~200 MB). If possible, run this download **before** the session.

In [ ]:
# --- Google Colab: mount Drive and download the catalog into it (run once) ---
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/GNN_school/data'
os.makedirs(DATA_DIR, exist_ok=True)

SNAP = 99   # snapshot 99 = z = 0 (present day). A lower snapshot = earlier time & smaller file.
URL  = f'https://users.flatironinstitute.org/~camels/Rockstar/CAMELS-SAM/LH/LH_35/Rockstar/out_{SNAP}.list'
CAMELS_FILE = os.path.join(DATA_DIR, f'CAMELS-SAM_LH35_out_{SNAP}.list')
BOX_SIZE = 100.0   # Mpc/h  (from the catalog header)

if os.path.exists(CAMELS_FILE) and os.path.getsize(CAMELS_FILE) > 0:
    print('Already in your Drive:', CAMELS_FILE)
else:
    print('Downloading out_%d.list into your Drive (a few minutes for z=0)...' % SNAP)
    !wget -q --show-progress -O "$CAMELS_FILE" "$URL"
print('File size: %.1f MB' % (os.path.getsize(CAMELS_FILE) / 1e6))

In [ ]:
# --- Load positions, masses, velocities from the Rockstar catalog ---
# Column layout (from the file header):
#   ID DescID Mvir Vmax Vrms Rvir Rs Np  X  Y  Z  VX VY VZ ...
#    0    1    2    3    4    5   6  7  8  9 10  11 12 13
import pandas as pd

df = pd.read_csv(CAMELS_FILE, sep=r'\s+', comment='#', header=None,
                 usecols=[2, 8, 9, 10, 11, 12, 13],
                 names=['Mvir', 'x', 'y', 'z', 'vx', 'vy', 'vz'])

position = df[['x', 'y', 'z']].to_numpy()
mass     = df['Mvir'].to_numpy()
velocity = df[['vx', 'vy', 'vz']].to_numpy()      # available for the feature experiment later
position = position % BOX_SIZE                      # wrap any tiny out-of-box coordinates

print('Halos loaded:', len(mass))
print('Position range (Mpc/h): %.2f to %.2f' % (position.min(), position.max()))
print('Mass range (Msun/h):    %.2e to %.2e' % (mass.min(), mass.max()))

## Step 2 — Explore the catalog

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(np.log10(mass), bins=40, color="steelblue")
ax[0].set_xlabel(r"$\log_{10}(M\,/\,M_\odot h^{-1})$"); ax[0].set_ylabel("N halos")
ax[0].set_title("Halo mass function")

ax[1].scatter(position[:, 0], position[:, 1], s=3, c=np.log10(mass), cmap="viridis")
ax[1].set_xlabel("x [Mpc/h]"); ax[1].set_ylabel("y [Mpc/h]")
ax[1].set_title("Halos (x-y projection)")
plt.tight_layout(); plt.show()

In [ ]:
# 3D view
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(position[:,0], position[:,1], position[:,2], s=3, c=np.log10(mass), cmap="viridis")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Dark-matter halos in the box")
plt.show()

## Step 3 — Build the graph (radius graph with periodic boundaries)

Two halos are connected if they lie closer than a **linking radius** `r`. Simulation boxes are
**periodic**, so a halo near one face is a neighbour of halos near the opposite face — we must
respect that. `scipy.spatial.cKDTree(boxsize=L)` handles periodicity for us.

`r` is a **hyperparameter**: too small → isolated nodes (no information flows); too large →
everything connects to everything (structure washed out). We will come back to this.


In [ ]:
# Keep well-resolved halos, and (for a responsive class) cap the number of nodes.
MASS_CUT  = 10 ** 12.0    # Msun/h ; raise to 10**12.5 or 10**13 for fewer, cleaner halos
MAX_HALOS = 8000          # subsample for speed; set to None to use every halo above the cut

keep = mass >= MASS_CUT
pos, m = position[keep], mass[keep]
if MAX_HALOS is not None and len(pos) > MAX_HALOS:
    sub = np.random.choice(len(pos), MAX_HALOS, replace=False)   # random subsample keeps the density structure
    pos, m = pos[sub], m[sub]
print("Halos after mass cut:", int(keep.sum()), "| used for the graph:", len(pos))

def build_edges(pos, r, box):
    "Periodic radius graph -> edge_index [2, E] (undirected, both directions)."
    tree = cKDTree(pos, boxsize=box)
    pairs = tree.query_pairs(r, output_type="ndarray")   # [P, 2], i<j
    if len(pairs) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    ei = np.concatenate([pairs.T, pairs.T[::-1]], axis=1) # add reverse edges
    return torch.tensor(ei, dtype=torch.long)

R_LINK = 5.0  # Mpc/h  <-- ~matches the local scale of the labels (5th-NN); see note above
edge_index = build_edges(pos, R_LINK, BOX_SIZE)
deg = np.bincount(edge_index[0].numpy(), minlength=len(pos))
print(f"r = {R_LINK} Mpc/h -> {edge_index.shape[1]} directed edges, "
      f"mean degree {deg.mean():.1f}, isolated nodes {(deg==0).sum()}")

In [ ]:
# visualise the graph (x-y projection); edges are subsampled so the plot stays legible
plt.figure(figsize=(7, 7))
und = edge_index.numpy()[:, ::2]                 # one direction per undirected edge
MAX_DRAW = 4000
sel = np.arange(und.shape[1])
if und.shape[1] > MAX_DRAW:
    sel = np.random.choice(und.shape[1], MAX_DRAW, replace=False)
for k in sel:
    a, b = und[0, k], und[1, k]
    if np.abs(pos[a] - pos[b]).max() < BOX_SIZE / 2:   # skip edges that wrap across the box
        plt.plot([pos[a, 0], pos[b, 0]], [pos[a, 1], pos[b, 1]], color="gray", lw=0.3, alpha=0.5)
plt.scatter(pos[:, 0], pos[:, 1], s=6, c="crimson", zorder=3)
plt.xlabel("x [Mpc/h]"); plt.ylabel("y [Mpc/h]")
plt.title(f"Halo graph (r={R_LINK} Mpc/h, {len(pos)} halos)")
plt.show()

## Step 4 — Define the labels: cluster vs void

We label each halo by its **local environment**. A simple, standard proxy is the distance to
the *k-th nearest neighbour*: small distance → crowded (cluster), large distance → empty (void).
We split at the median so the two classes are balanced.

> **Important — no label leakage:** the label is built from local density, so we must **not**
> feed density (or neighbour counts) as a node feature. We give the network only the halo
> **mass**; it has to infer the environment from the *graph structure*. That is the lesson.


In [ ]:
K_ENV = 5
tree = cKDTree(pos, boxsize=BOX_SIZE)
dknn, _ = tree.query(pos, k=K_ENV + 1)      # +1 because the first neighbour is the point itself
env_dist = dknn[:, -1]                        # distance to k-th neighbour
labels = (env_dist < np.median(env_dist)).astype(np.int64)  # 1 = cluster, 0 = void

print("cluster halos:", labels.sum(), " void halos:", (labels == 0).sum())

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(pos[labels==1,0], pos[labels==1,1], pos[labels==1,2], s=6, c="crimson", label="cluster")
ax.scatter(pos[labels==0,0], pos[labels==0,1], pos[labels==0,2], s=6, c="royalblue", label="void")
ax.legend(); ax.set_title("Node labels: cluster vs void"); plt.show()

## Step 5 — Assemble the PyG graph and split the nodes

Node feature `x` = normalised log-mass (shape `[N, 1]`). We create train / val / test **masks**
over the nodes (transductive node classification: one graph, labelled subset of nodes).


In [ ]:
logm = np.log10(m)
x = torch.tensor((logm - logm.mean()) / logm.std(), dtype=torch.float32).view(-1, 1)
y = torch.tensor(labels, dtype=torch.long)

N = len(m)
perm = np.random.permutation(N)
n_tr, n_va = int(0.6 * N), int(0.2 * N)
train_mask = torch.zeros(N, dtype=torch.bool); train_mask[perm[:n_tr]] = True
val_mask   = torch.zeros(N, dtype=torch.bool); val_mask[perm[n_tr:n_tr+n_va]] = True
test_mask  = torch.zeros(N, dtype=torch.bool); test_mask[perm[n_tr+n_va:]] = True

data = Data(x=x, edge_index=edge_index, y=y,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask).to(device)
print(data)

## Step 6 — Define the GNN

Which message-passing layer we pick matters a lot here. The label is *local density*, and the
feature that carries it is **how many neighbours a halo has** (its degree). But a plain **GCN**
uses symmetric normalisation ($\hat D^{-1/2}\hat A\hat D^{-1/2}$) that **divides the degree
signal away** — so a GCN on mass-only features gets stuck around ~70–75%.

We instead use **GraphSAGE with *sum* aggregation** (`aggr="add"`): summing over neighbours keeps
the neighbour **count**, i.e. the local density — exactly the signal the task needs. Combined with
a linking radius `r` close to the label's scale, this reaches **~90%+**. (We still feed only the
halo mass as input — the environment is learned from the graph structure, not handed to the model.)

> Teaching point: this is *why* the GNN zoo matters. Try switching `SAGEConv` back to `GCNConv`
> below and watch the accuracy fall — the layer choice, not the tuning, is what unlocks the task.

In [ ]:
class GraphSAGE(torch.nn.Module):
    """Sum-aggregating GraphSAGE: keeps neighbour COUNT (= local density), the signal the
    cluster/void label depends on. (Swap SAGEConv->GCNConv to see it drop back to ~70%.)"""
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden, aggr="add")
        self.conv2 = SAGEConv(hidden, hidden, aggr="add")
        self.conv3 = SAGEConv(hidden, out_dim, aggr="add")

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        return x                       # raw logits

model = GraphSAGE(in_dim=1, hidden=64, out_dim=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print(model)

## Step 7 — Train

In [ ]:
def accuracy(logits, y, mask):
    pred = logits.argmax(dim=1)
    return (pred[mask] == y[mask]).float().mean().item()

history = {"loss": [], "train": [], "val": []}
for epoch in range(1, 201):
    model.train(); optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        tr, va = accuracy(out, data.y, data.train_mask), accuracy(out, data.y, data.val_mask)
    history["loss"].append(loss.item()); history["train"].append(tr); history["val"].append(va)
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.3f} | train {tr:.3f} | val {va:.3f}")

plt.plot(history["train"], label="train acc")
plt.plot(history["val"], label="val acc")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

## Step 8 — Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=1).cpu().numpy()
y_np = data.y.cpu().numpy(); tm = data.test_mask.cpu().numpy()

print("Test accuracy: %.3f" % (pred[tm] == y_np[tm]).mean())
print(classification_report(y_np[tm], pred[tm], target_names=["void", "cluster"]))
print("Confusion matrix:\n", confusion_matrix(y_np[tm], pred[tm]))

In [ ]:
# where does the model get it right / wrong?  (test nodes, x-y projection)
correct = (pred == y_np) & tm
wrong   = (pred != y_np) & tm
plt.figure(figsize=(7, 6))
plt.scatter(pos[correct,0], pos[correct,1], s=8, c="seagreen", label="correct")
plt.scatter(pos[wrong,0],   pos[wrong,1],   s=20, c="red", marker="x", label="wrong")
plt.xlabel("x [Mpc/h]"); plt.ylabel("y [Mpc/h]"); plt.legend()
plt.title("Test-set predictions"); plt.show()

## Step 9 — Experiments (your turn)

Try these and watch how the test accuracy and the graph change:

1. **Linking radius `r`.** Rebuild the graph with `R_LINK` = 1, 2, 4, 8 Mpc/h. What happens at
   the extremes (too many isolated nodes / a fully connected blob)? This is *over-smoothing* in action.
2. **Depth.** Add / remove `GCNConv` layers. Do very deep GCNs help or hurt? (over-smoothing again.)
3. **Attention.** Swap `GCNConv` for `GATConv` (a small starter is below). Does attention help?
4. **Features.** Add a second node feature (e.g. `|velocity|` if you loaded real data) and compare.
5. **A fair baseline.** Train a plain MLP on the node features *only* (no edges). How much does the
   graph structure actually buy you? This is the experiment that proves the point of GNNs.


In [ ]:
# Starter: a GAT variant for experiment 3
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim, heads=4):
        super().__init__()
        self.g1 = GATConv(in_dim, hidden, heads=heads)
        self.g2 = GATConv(hidden * heads, out_dim, heads=1)
    def forward(self, x, edge_index):
        x = F.elu(self.g1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.g2(x, edge_index)

# model = GAT(1, 32, 2).to(device)   # uncomment and re-run the training cell

## Conclusion

You built a graph from a halo catalog, defined a physical node label (cluster vs void), and
trained a GNN that recovers a halo's **environment from graph structure alone**.

In **Hands-on 2** we move from *node-level* to *graph-level* learning: each simulation box becomes
one graph, and we train a GNN to **regress a cosmological parameter (Ω_m)** for the whole box —
a small, hands-on version of likelihood-free cosmological inference.
